# 第 10 章 · LoRA(精简版)

> 本文是 [ch10.ipynb](./ch10.ipynb) 的浓缩版。只保留核心公式和关键数字,方便快速复习。

## 核心公式

$$W' = W + \Delta W = W + B \cdot A$$

- $W \in \mathbb{R}^{d \times d}$:冻结
- $A \in \mathbb{R}^{r \times d}$:高斯初始化,可训练
- $B \in \mathbb{R}^{d \times r}$:**零初始化**,可训练
- $r = 16 \ll d = 768$

## 参数量

| | 全参 | LoRA |
|---|---|---|
| 单层 768×768 | 589,824 | 24,576(4.17%) |
| 整个模型 | 63.9M(100%) | 0.39M(0.61%) |
| 保存文件 | ~131 MB | ~0.8 MB |

## minimind 关键设计

```python
# LoRA 模块:A 高斯 + B 零
class LoRA(nn.Module):
    def __init__(self, in_features, out_features, rank):
        self.A = nn.Linear(in_features, rank, bias=False)
        self.B = nn.Linear(rank, out_features, bias=False)
        self.A.weight.data.normal_(mean=0.0, std=0.02)
        self.B.weight.data.zero_()           # ← 零初始化

# 挂载:只挂方形 Linear (in == out)
def apply_lora(model, rank=16):
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and module.in_features == module.out_features:
            # → 挂 q_proj (768→768) + o_proj (768→768)
            # → 跳过 k/v_proj (768→384, GQA) + FFN + lm_head
            ...
```

## 训练差异

| | 全参 SFT | LoRA |
|---|---|---|
| lr | 1e-5 | 1e-4(10×) |
| 可训练参数 | 100% | 0.61% |
| epochs | 3-5 | 10 |
| torch.compile | ✓ | ✗ |
| from_weight | pretrain | **full_sft** |

## 三个操作

| 操作 | 作用 | 文件 |
|---|---|---|
| `save_lora` | 存 A+B | ~0.8 MB |
| `load_lora` | 加载到 base model | 2 个文件 |
| `merge_lora` | $W += BA$ | ~131 MB(完整) |

## 下一步

LoRA 掌握后 → 第 11 章:推理工程